### Importing libraries

In [ ]:
from pathlib import Path
import bioread
import numpy as np
import pandas as pd
from scipy import signal
from helpers import (fir_bandpass, comb_band, integration_moving_rms,
                    read_acq_files, show_channels, choose_indexes_per_file,
                    get_channel_idx, save_processed_emg_h5, express_as_percent_mvc)

### Path to the files

In [2]:
folder_root_path = r"data/"     # r"I:/Research/Shares/dover_research/Div_fEMG/"

In [3]:
study_id = "TEST"

In [4]:
files_suffix = {
    "APP": Path(f"{folder_root_path}{study_id}_fEMG_div_2APP_raw.acq"),
    "PsychoPy": Path(f"{folder_root_path}{study_id}_fEMG_div_3PsychoPy_raw.acq"),
    "SlowRate": Path(f"{folder_root_path}{study_id}_fEMG_div_4SlowRate_raw.acq"),
}

In [5]:
file_type_list = ["APP", "PsychoPy", "SlowRate"]

### Check for the path of the files

In [6]:
if pd.DataFrame([
    {"task": task, "path": str(path.resolve()), "exists": path.is_file()}
    for task, path in files_suffix.items()
])['exists'].all():
    print('All file path exits')
else:
    raise FileNotFoundError('Not all file path exist, please check rootfolder and file paths')

All file path exits


### Read files

In [7]:
data = read_acq_files(files_suffix)

In [8]:
data

{'APP': AcqKnowledge file (rev 133): 8 channels, 2000.0 samples/sec,
 'PsychoPy': AcqKnowledge file (rev 133): 8 channels, 2000.0 samples/sec,
 'SlowRate': AcqKnowledge file (rev 133): 5 channels, 2000.0 samples/sec}

In [9]:
data_df = show_channels(data)

In [10]:
data_df

,task,index,order_num,name,units,rate_hz,samples
0,APP,0,1,ZM - FIR,mV,2000.0,1262646
1,APP,1,2,ZM - Comb,mV,2000.0,1262646
2,APP,2,3,ZM - EMG100C,mV,2000.0,1262646
3,APP,3,4,CS - EMG100C,mV,2000.0,1262646
4,APP,4,5,ZM - Int,mV,2000.0,1262646
5,APP,5,6,CS - FIR,mV,2000.0,1262646
6,APP,6,7,CS - Comb,mV,2000.0,1262646
7,APP,7,8,CS - Integrate,mV,2000.0,1262646
8,PsychoPy,0,1,ZM - FIR,mV,2000.0,1137980
9,PsychoPy,1,2,ZM - Comb,mV,2000.0,1137980


In [17]:
for file_type in file_type_list:
    print(f"File type: {file_type}")
    app_idx = choose_indexes_per_file(data_df, file_type)
    ZM = {}
    CS = {}
    reference_channel = None
    sampling_rate = None

    for i, muscle in enumerate(["ZM", "CS"]):
        reference_channel = get_channel_idx(data, file_type, app_idx, muscle)
        sampling_rate = reference_channel.samples_per_second
        print(f"{i} - {reference_channel} - {sampling_rate} Hz")
        fir = fir_bandpass(np.asarray(reference_channel.data, dtype=float), sampling_rate)
        comb = comb_band(fir, sampling_rate, base_hz=60.0, q=5)
        rms = integration_moving_rms(comb, window_samples=1000)

        if muscle == "ZM":
            ZM["rms"] = rms
            ZM["percent"] = express_as_percent_mvc(rms, 0.101)
        if muscle == "CS":
            CS["rms"] = rms
            CS["percent"] = express_as_percent_mvc(rms, 0.083)

    source_path = Path(files_suffix[file_type])

    # Use each recording's filename and a fresh folder for this batch.
    output_path = Path("output") / "batch_01" / (
        f"{source_path.stem.removesuffix('_raw')}_filter.h5"
    )
    saved_path = save_processed_emg_h5(
        output_path,
        recording=data[file_type],  # Original PsychoPy recording loaded with bioread
        source_path=source_path,
        sampling_rate=sampling_rate,
        zm_rms=ZM["rms"],
        zm_percent=ZM["percent"],
        cs_rms=CS["rms"],
        cs_percent=CS["percent"],
        zm_mvc=0.101,
        cs_mvc=0.083,
    )
    

        

File type: APP
0 - Channel ZM - EMG100C: 1262646 samples, 2000.0 samples/sec, loaded: True - 2000.0 Hz
1 - Channel CS - EMG100C: 1262646 samples, 2000.0 samples/sec, loaded: True - 2000.0 Hz
File type: PsychoPy
0 - Channel ZM - EMG100C: 1137980 samples, 2000.0 samples/sec, loaded: True - 2000.0 Hz
1 - Channel CS - EMG100C: 1137980 samples, 2000.0 samples/sec, loaded: True - 2000.0 Hz
File type: SlowRate
0 - Channel ZM - EMG100C: 805412 samples, 2000.0 samples/sec, loaded: True - 2000.0 Hz
1 - Channel CS - EMG100C: 805412 samples, 2000.0 samples/sec, loaded: True - 2000.0 Hz


In [19]:
from pathlib import Path
import json
from bs4 import BeautifulSoup
import h5py
import pandas as pd
from IPython.display import display

output_dir = Path("output/batch_01")

for file_type, source in files_suffix.items():
    source = Path(source)
    path = output_dir / f"{source.stem.removesuffix('_raw')}_filter.h5"

    print(f"\n--- {file_type}: {path.name} ---")

    if not path.is_file():
        print("File not found.")
        continue

    with h5py.File(path, "r") as h5:
        metadata = json.loads(h5["metadata/source"].asstr()[()])
        labels = h5["channel_order"].asstr()[:]
        signals = h5["signals"]

        print("Source recording:", metadata["filename"])
        if metadata["filename"] != source.name:
            raise ValueError(f"{path.name}: saved source does not match {source.name}")

        display(pd.DataFrame([
            {
                "channel": label,
                "samples": signals[label].shape[0],
                "sampling_rate_hz": signals[label].attrs["sampling_rate_hz"],
                "units": signals[label].attrs["units"],
            }
            for label in labels
        ]))

        print("First five samples:")
        display(pd.DataFrame({label: signals[label][:5] for label in labels}))

        print("Journal:")

        soup = BeautifulSoup(metadata.get("journal") or "", "html.parser")
        for tag in soup(["head", "style", "script"]):
            tag.decompose()

        paragraphs = [p.get_text(" ", strip=True) for p in soup.find_all("p")]
        journal_text = "\n".join(text for text in paragraphs if text)

        if not journal_text:
            journal_text = soup.get_text("\n", strip=True)

        print(journal_text or "(No journal text in the source recording.)")

        print("Event markers:")
        markers = metadata.get("event_markers", [])
        if markers:
            display(pd.DataFrame(markers))
        else:
            print("(No event markers in the source recording.)")


--- APP: TEST_fEMG_div_2APP_filter.h5 ---
Source recording: TEST_fEMG_div_2APP_raw.acq


,channel,samples,sampling_rate_hz,units
0,ZM mv,1262646,2000.0,mV
1,ZM %,1262646,2000.0,% MVC
2,CS mv,1262646,2000.0,mV
3,CS %,1262646,2000.0,% MVC


First five samples:


,ZM mv,ZM %,CS mv,CS %
0,0.038819,38.434888,0.062851,75.724411
1,0.063490,62.861353,0.133418,160.744813
2,0.077270,76.504516,0.192414,231.824267
3,0.109030,107.950429,0.276748,333.431585
4,0.148103,146.636317,0.353487,425.888393


Journal:
Participant Number: 1011
Notes:
Event Summary
Index	Time	Type	Channel	Label	Date Created
0	0.000 ns	Append	Global	Segment 1	Wed Jun 29 09:42:39 2022
1	15.153 sec	Default	Global	VidCue Wed Jun 29 2022 09:42:54	Wed Jun 29 09:42:54 2022
2	3.251 min	Default	Global	Star Wed Jun 29 2022 09:45:54	Wed Jun 29 09:45:54 2022
Modification History for CH1
Transformation Name: FIR Filter
Channel: CH1
Date Time: Thu Oct 15 2026 - 12:08:31
Filter Type: Band Pass
Window Type: Blackman -61 dB windowing
Low Frequency Cutoff: 28
High Frequency Cutoff: 500
Number of Coefficients: 285
Starting sample position to transform: 1
Ending sample position to transform: 1262646
Modification History for CH2
Transformation Name: Comb Band Stop Filter
Channel: CH2
Date Time: Thu Oct 15 2026 - 12:13:35
Base Frequency: 60
Q: 5
Overharmonics: All harmonics
starting sample position to transform: 1
ending sample position to transform: 1262646
Modification History for CH5
Transformation Name: Integrate Average Over 

,text,time_index,sample_index,type_code,channel_number
0,Segment 1,0.0000,0,apnd,None
1,VidCue Wed Jun 29 2022 09:42:54,15.1530,30306,defl,None
2,Star Wed Jun 29 2022 09:45:54,195.0575,390115,defl,None


... (8 outputs left out)

First five samples:


,ZM mv,ZM %,CS mv,CS %
0,0.073519,72.791213,0.051653,62.233107
1,0.084012,83.180088,0.055713,67.123642
2,0.059674,59.083539,0.043321,52.193663
3,0.062360,61.742720,0.057144,68.847757
4,0.079325,78.539205,0.075984,91.547153


Journal:
Participant Number:
Notes:
Modification History for CH1
Transformation Name: FIR Filter
Channel: CH1
Date Time: Thu Oct 15 2026 - 14:45:35
Filter Type: Band Pass
Window Type: Blackman -61 dB windowing
Low Frequency Cutoff: 28
High Frequency Cutoff: 500
Number of Coefficients: 285
Starting sample position to transform: 1
Ending sample position to transform: 805412
Modification History for CH2
Transformation Name: Comb Band Stop Filter
Channel: CH2
Date Time: Thu Oct 15 2026 - 14:47:25
Base Frequency: 60
Q: 5
Overharmonics: All harmonics
starting sample position to transform: 1
ending sample position to transform: 805412
Modification History for CH5
Transformation Name: Integrate Average Over Samples
Channel: CH5
Date Time: Thu Oct 15 2026 - 14:49:41
Samples: 1000
Parameter: Root Mean Square
remove baseline from data: no
starting sample position to analyze: 1
ending sample position to analyze: 805412
Event markers:


,text,time_index,sample_index,type_code,channel_number
0,Segment 1,0.0000,0,apnd,None
1,Star Wed Jun 29 2022 10:06:19,24.7585,49517,defl,None
